# Support Vector Machine (SVM)

**Course**: CMOR 438 / INDE 577 — Data Science & Machine Learning  
**Dataset**: International Football Results (1872–present)  
**Author**: Neriah29  

---

## What is an SVM?

Every classifier we've built finds *a* decision boundary. SVM finds the **best possible** one — the boundary that is maximally distant from both classes simultaneously.

### The Maximum Margin Principle

Among all lines that correctly separate two classes, the one with the **largest margin** (gap between classes) is the most robust to new data. SVM finds exactly that line.

```
         Class 0    |    Class 1
           o        |        x
             o    --+--    x          ← margin
           o        |        x
                    ↑
              Decision boundary
              (maximum margin)
```

The points closest to the boundary on each side are the **support vectors** — they are the only training points that define the boundary. Remove any other point and the boundary doesn't change.

### Soft Margin — Handling Noise

Real data has overlap. The **C parameter** controls the tradeoff:
- **Large C**: strict — few misclassifications allowed, tight margin
- **Small C**: relaxed — allows misclassifications, wider margin

### The Kernel Trick

When classes aren't linearly separable, the **kernel function** implicitly maps data into a higher-dimensional space where a linear boundary does separate them — without ever computing the transformation explicitly.

- **Linear kernel**: no transformation
- **RBF kernel**: exp(-γ||x-x'||²) — maps to infinite dimensions, separates almost anything

### Important: SVM is not probabilistic

Unlike Logistic Regression, SVM outputs a **distance from the boundary** (decision score), not a probability. We can approximate probabilities by passing the score through a sigmoid, but this is less principled than Logistic Regression's output.


---
## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

import sys
sys.path.insert(0, '../../src')
from football_ml.supervised_learning.svm import SVM

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
SEED = 42

---
## 1. Visualise the Maximum Margin Concept

In [ ]:
rng = np.random.default_rng(0)
X0_toy = rng.normal(loc=[-2, -1], scale=0.5, size=(20, 2))
X1_toy = rng.normal(loc=[2,  1],  scale=0.5, size=(20, 2))
X_toy  = np.vstack([X0_toy, X1_toy])
y_toy  = np.array([0]*20 + [1]*20)

# Fit linear SVM
from sklearn.preprocessing import StandardScaler as SK_Scaler
sc = SK_Scaler()
X_toy_sc = sc.fit_transform(X_toy)

svm_toy = SVM(kernel='linear', C=1.0, max_iter=500).fit(X_toy_sc, y_toy)

# Plot
fig, ax = plt.subplots(figsize=(7, 5))
xx, yy = np.meshgrid(
    np.linspace(X_toy_sc[:,0].min()-0.5, X_toy_sc[:,0].max()+0.5, 300),
    np.linspace(X_toy_sc[:,1].min()-0.5, X_toy_sc[:,1].max()+0.5, 300),
)
Z = svm_toy.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

ax.contourf(xx, yy, Z, levels=[-999, 0, 999], alpha=0.2, colors=['#E87272', '#4878CF'])
ax.contour(xx, yy, Z, levels=[-1, 0, 1], colors=['#E87272', 'black', '#4878CF'],
           linewidths=[1.5, 2.5, 1.5], linestyles=['--', '-', '--'])

ax.scatter(X_toy_sc[y_toy==0, 0], X_toy_sc[y_toy==0, 1],
           c='#E87272', edgecolors='black', s=50, label='Class 0')
ax.scatter(X_toy_sc[y_toy==1, 0], X_toy_sc[y_toy==1, 1],
           c='#4878CF', edgecolors='black', s=50, label='Class 1')

# Highlight support vectors
ax.scatter(svm_toy.support_vectors_[:, 0], svm_toy.support_vectors_[:, 1],
           s=200, facecolors='none', edgecolors='black', linewidth=2,
           label=f'Support vectors ({svm_toy.n_support_})')

ax.set_title('SVM — Maximum Margin Decision Boundary', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

The solid black line is the decision boundary. The dashed lines mark the margin edges — the SVM maximizes the gap between them. Circled points are the **support vectors** — the only training samples that actually define the boundary.

---
## 2. RBF Kernel — Non-linear Boundary

In [ ]:
from sklearn.datasets import make_circles
X_circ, y_circ = make_circles(n_samples=150, noise=0.1, factor=0.4, random_state=SEED)
sc2 = SK_Scaler()
X_circ_sc = sc2.fit_transform(X_circ)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, kernel, title in zip(axes,
                              ['linear', 'rbf'],
                              ['Linear Kernel (fails on circles)', 'RBF Kernel (solves circles)']):
    m = SVM(kernel=kernel, C=1.0, max_iter=300).fit(X_circ_sc, y_circ)
    xx, yy = np.meshgrid(
        np.linspace(-2, 2, 200), np.linspace(-2, 2, 200)
    )
    Z = m.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X_circ_sc[:,0], X_circ_sc[:,1], c=y_circ,
               cmap='RdBu', edgecolors='white', s=25)
    ax.set_title(f'{title}\nAccuracy: {m.score(X_circ_sc, y_circ):.1%}',
                 fontsize=11, fontweight='bold')

plt.suptitle('Kernel Comparison — Circular Data', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

The linear kernel can't separate concentric circles — no straight line can. The RBF kernel implicitly maps the data to a higher-dimensional space where the circles become linearly separable.

---
## 3. Load & Engineer Features

In [ ]:
df = pd.read_csv('../../data/results.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
df['home_win'] = (df['home_score'] > df['away_score']).astype(int)

def compute_team_rolling_stats(df, window=10):
    home_log = df[['date', 'home_team', 'home_score', 'away_score']].copy()
    home_log.columns = ['date', 'team', 'scored', 'conceded']
    away_log = df[['date', 'away_team', 'away_score', 'home_score']].copy()
    away_log.columns = ['date', 'team', 'scored', 'conceded']
    team_log = pd.concat([home_log, away_log]).sort_values('date').reset_index(drop=True)
    team_log['rolling_scored'] = (
        team_log.groupby('team')['scored']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    team_log['rolling_conceded'] = (
        team_log.groupby('team')['conceded']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    return team_log.drop_duplicates(subset=['date', 'team'], keep='last').set_index(['date', 'team'])

team_stats = compute_team_rolling_stats(df)

def get_stat(row, team_col, stat_col):
    try:
        return team_stats.loc[(row['date'], row[team_col]), stat_col]
    except KeyError:
        return np.nan

df['home_goals_rolling']    = df.apply(lambda r: get_stat(r, 'home_team', 'rolling_scored'), axis=1)
df['home_conceded_rolling'] = df.apply(lambda r: get_stat(r, 'home_team', 'rolling_conceded'), axis=1)
df['away_goals_rolling']    = df.apply(lambda r: get_stat(r, 'away_team', 'rolling_scored'), axis=1)
df['away_conceded_rolling'] = df.apply(lambda r: get_stat(r, 'away_team', 'rolling_conceded'), axis=1)

home_wins = df.groupby('home_team').apply(lambda g: (g['home_score'] > g['away_score']).mean()).rename('home_win_rate')
away_wins = df.groupby('away_team').apply(lambda g: (g['away_score'] > g['home_score']).mean()).rename('away_win_rate')
df = df.join(home_wins, on='home_team').join(away_wins, on='away_team')
df['neutral'] = df['neutral'].astype(int)

feature_cols = [
    'home_goals_rolling', 'away_goals_rolling',
    'home_conceded_rolling', 'away_conceded_rolling',
    'home_win_rate', 'away_win_rate', 'neutral'
]
df_clean = df[feature_cols + ['home_win']].dropna()

# Use a subset for speed — SVM scales as O(n²) with dataset size
df_sample = df_clean.sample(n=3000, random_state=SEED)
print(f'Using {len(df_sample)} matches for SVM (full dataset too slow for from-scratch SMO)')

---
## 4. Prepare Data

In [ ]:
X = df_sample[feature_cols].values
y = df_sample['home_win'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# Scaling is critical for SVM — distances drive everything
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train_sc.shape} | Test: {X_test_sc.shape}')

---
## 5. Train & Evaluate

In [ ]:
model = SVM(kernel='rbf', C=1.0, gamma='scale', max_iter=500).fit(X_train_sc, y_train)

print(f'Support vectors:  {model.n_support_}')
print(f'Train accuracy:   {model.score(X_train_sc, y_train):.3f}')
print(f'Test  accuracy:   {model.score(X_test_sc,  y_test):.3f}')

---
## 6. Effect of C

In [ ]:
C_values = [0.01, 0.1, 1.0, 10.0]
train_scores, test_scores, n_svs = [], [], []

for C in C_values:
    m = SVM(kernel='rbf', C=C, max_iter=300).fit(X_train_sc, y_train)
    train_scores.append(m.score(X_train_sc, y_train))
    test_scores.append(m.score(X_test_sc,   y_test))
    n_svs.append(m.n_support_)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(C_values, train_scores, 'o-', label='Train', color='#4878CF', linewidth=2)
axes[0].plot(C_values, test_scores,  'o-', label='Test',  color='#E87272', linewidth=2)
axes[0].set_xscale('log')
axes[0].set_xlabel('C (log scale)', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Accuracy vs C', fontsize=12, fontweight='bold')
axes[0].legend()

axes[1].plot(C_values, n_svs, 'o-', color='#2ecc71', linewidth=2)
axes[1].set_xscale('log')
axes[1].set_xlabel('C (log scale)', fontsize=12)
axes[1].set_ylabel('Number of Support Vectors', fontsize=12)
axes[1].set_title('Support Vectors vs C', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

### What to notice

- **Small C**: more support vectors (wider, more relaxed margin), lower train accuracy but potentially better generalization
- **Large C**: fewer support vectors (tighter margin), higher train accuracy but higher overfitting risk

The number of support vectors is inversely related to C — the stricter the margin, the fewer points define it.

---
## 7. Confusion Matrix & ROC

In [ ]:
y_pred = model.predict(X_test_sc)
probs  = model.predict_proba(X_test_sc)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred: 0', 'Pred: 1'],
            yticklabels=['True: 0', 'True: 1'], ax=axes[0])
axes[0].set_title('Confusion Matrix', fontsize=12, fontweight='bold')

fpr, tpr, _ = roc_curve(y_test, probs)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='#4878CF', linewidth=2, label=f'SVM RBF (AUC={roc_auc:.3f})')
axes[1].plot([0,1],[0,1],'gray',linestyle='--',linewidth=1)
axes[1].set_xlabel('False Positive Rate', fontsize=11)
axes[1].set_ylabel('True Positive Rate', fontsize=11)
axes[1].set_title('ROC Curve', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=['Draw/Away', 'Home Win']))

---
## 8. Discussion — Does SVM Suit Football?

### What it does well
- **Maximum margin**: theoretically the most robust linear classifier — the widest possible gap between classes
- **Kernel trick**: RBF kernel can find complex non-linear boundaries
- **Works well in high dimensions**: SVM excels when you have many features relative to samples
- **Sparse solution**: only support vectors matter — elegant and memory-efficient

### What it struggles with
1. **Scales as O(n²) or O(n³)**: with 40,000+ matches, training becomes slow — we sampled 3,000 here
2. **No native probability output**: the sigmoid approximation is less principled than Logistic Regression
3. **Sensitive to C and gamma**: requires careful tuning
4. **Football noise**: the maximum margin concept assumes a clear gap between classes — football data has enormous overlap

### Where SVM truly shines

SVM was the state-of-the-art classifier before deep learning took over. It's particularly powerful for text classification, image recognition with handcrafted features, and bioinformatics — domains with high-dimensional, clean data. For messy, noisy tabular data like football results, ensemble methods (Random Forest, Gradient Boosting) tend to outperform it.

---
## Summary

| | |
|---|---|
| **Algorithm type** | Maximum margin binary classifier |
| **Output** | Decision score (signed distance from boundary) |
| **Probability** | Approximate (sigmoid on score) |
| **Key parameter** | C — margin strictness tradeoff |
| **Kernels** | Linear, RBF (others possible) |
| **Needs scaling** | Yes — critical |
| **Key concept** | Maximum margin, support vectors, kernel trick |
| **Football suitability** | Moderate — powerful but slow and noisy data |
